In [1]:
import pandas as pd 
pd.read_csv('data/new_data/new_data.csv')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,prediction,probability,Churn,timestamp,label_timestamp
0,75920-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,NaN,2026-04-14T05:25:04.846702,NaN
1,7590-VHVsdaEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,Yes,2026-04-14T06:19:41.011236,2026-04-14T07:08:38.186056
2,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,Yes,2026-04-14T06:19:53.073309,2026-04-14T06:20:29.616893
3,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,0,0.152464,No,2026-04-14T06:19:53.097699,2026-04-14T07:08:38.187055
4,75190-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,Yes,2026-04-14T07:03:29.718776,2026-04-14T07:07:58.266746
5,55575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,0,0.152464,No,2026-04-14T07:03:29.726738,2026-04-14T07:07:31.741249
6,55571d5-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,0,0.152464,No,2026-04-14T07:03:37.277951,2026-04-14T07:07:17.026232
7,7512w90-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,NaN,2026-04-14T07:03:42.991139,NaN
8,7512eqew90-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,1,0.935113,Yes,2026-04-14T07:03:47.782482,2026-04-14T07:07:20.557135
9,55571das5-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,0,0.152464,No,2026-04-14T07:03:47.805753,2026-04-14T07:06:38.849264


In [2]:
import pandas as pd
import cloudpickle
from pathlib import Path

def inspect_transformed_data(model_path, data_path):
    """
    Показывает данные после каждого шага трансформации
    """
    # Загружаем модель и данные
    with open(model_path, 'rb') as f:
        pipeline = cloudpickle.load(f)
    
    df = pd.read_csv(data_path)
    df_labeled = df[df['Churn'].notna()].copy()
    
    # Подготавливаем сырые данные (как при обучении)
    feature_columns = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges'
    ]
    
    X_raw = df_labeled[feature_columns].copy()
    
    print("=" * 80)
    print("DATA TRANSFORMATION INSPECTION")
    print("=" * 80)
    
    # Применяем каждый шаг пайплайна по очереди
    X_transformed = X_raw.copy()
    
    for name, step in pipeline.named_steps.items():
        print(f"\n--- AFTER {name.upper()} ---")
        print(f"Shape: {X_transformed.shape}")
        print(f"Columns: {list(X_transformed.columns)}")
        print(f"First row sample:\n{X_transformed.iloc[0]}")
        print("-" * 40)
        
        # Применяем трансформацию
        X_transformed = step.transform(X_transformed)
    
    print(f"\n--- FINAL TRANSFORMED DATA (what model sees) ---")
    print(f"Shape: {X_transformed.shape}")
    print(f"Columns: {list(X_transformed.columns)}")
    print(f"First row sample:\n{X_transformed.iloc[0]}")
    
    return X_transformed

# Использование
X_final = inspect_transformed_data(
    'models/full_churn_pipeline_retrained_cloud.pkl',
    'data/new_data/new_data.csv'
)

DATA TRANSFORMATION INSPECTION

--- AFTER TOTALCHARGES_CLEANER ---
Shape: (10, 19)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
First row sample:
gender                        Female
SeniorCitizen                      0
Partner                          Yes
Dependents                        No
tenure                             1
PhoneService                      No
MultipleLines       No phone service
InternetService                  DSL
OnlineSecurity                    No
OnlineBackup                     Yes
DeviceProtection                  No
TechSupport                       No
StreamingTV                       No
StreamingMovies                   No
Contract              Month-to-month
PaperlessBilling                 Y

AttributeError: 'CatBoostClassifier' object has no attribute 'transform'

In [9]:
import cloudpickle
from pathlib import Path

def get_model_feature_order(model_path):
    """
    Получить порядок колонок, который ожидает CatBoost модель
    """
    with open(model_path, 'rb') as f:
        pipeline = cloudpickle.load(f)
    
    # Находим CatBoost модель в пайплайне
    catboost_model = None
    for name, step in pipeline.named_steps.items():
        if 'model' in name or 'catboost' in name.lower():
            catboost_model = step
            break
    
    if catboost_model is None:
        print("CatBoost model not found in pipeline")
        return None
    
    # Способ 1: feature_names_ (если есть)
    if hasattr(catboost_model, 'feature_names_'):
        print("Feature names from model:")
        for i, name in enumerate(catboost_model.feature_names_):
            print(f"  {i}: {name}")
        return catboost_model.feature_names_
    
    # Способ 2: Получить из пайплайна после трансформаций
    # Применяем все трансформации к sample данных
    import pandas as pd
    import numpy as np
    
    # Создаем sample данных со всеми признаками
    sample_data = pd.DataFrame({
        'gender': ['Female'],
        'SeniorCitizen': [0],
        'Partner': ['Yes'],
        'Dependents': ['No'],
        'tenure': [1],
        'PhoneService': ['No'],
        'MultipleLines': ['No phone service'],
        'InternetService': ['DSL'],
        'OnlineSecurity': ['No'],
        'OnlineBackup': ['Yes'],
        'DeviceProtection': ['No'],
        'TechSupport': ['No'],
        'StreamingTV': ['No'],
        'StreamingMovies': ['No'],
        'Contract': ['Month-to-month'],
        'PaperlessBilling': ['Yes'],
        'PaymentMethod': ['Electronic check'],
        'MonthlyCharges': [29.85],
        'TotalCharges': ['29.85']
    })
    
    # Применяем все трансформации КРОМЕ модели
    transform_pipeline = pipeline[:-1]
    X_transformed = transform_pipeline.transform(sample_data)
    
    print("\nFeature order after transformations (what CatBoost sees):")
    if hasattr(X_transformed, 'columns'):
        for i, col in enumerate(X_transformed.columns):
            print(f"  {i}: {col}")
        return X_transformed.columns.tolist()
    else:
        print(f"  Result is numpy array with {X_transformed.shape[1]} features")
        print(f"  Feature names not available")
        return None

# Использование
feature_order = get_model_feature_order('models/full_churn_pipeline_retrained_cloud.pkl')

Feature names from model:
  0: SeniorCitizen
  1: PhoneService
  2: Partner
  3: gender
  4: tenure
  5: PaperlessBilling
  6: Dependents
  7: avg_monthly_spend
  8: tenure_group
  9: has_internet
  10: has_addons
  11: risky_contract
  12: auto_payment
  13: MultipleLines_No phone service
  14: MultipleLines_Yes
  15: InternetService_Fiber optic
  16: InternetService_No
  17: PaymentMethod_Credit card (automatic)
  18: PaymentMethod_Electronic check
  19: PaymentMethod_Mailed check


In [29]:
model_path = 'models/full_churn_pipeline_retrained_cloud.pkl'
with open(model_path, 'rb') as f:
    pipeline = cloudpickle.load(f)

In [31]:
df = pd.read_csv('data/new_data/new_data.csv')
df_labeled = df[df['Churn'].notna()].copy()
    
feature_columns = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges'
    ]
    
X_raw = df_labeled[feature_columns].copy()

In [33]:
pipeline.predict(X_raw)

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 2 should be feature with name PhoneService (found Partner).

In [35]:
import pandas as pd
import cloudpickle
from pathlib import Path

# Загружаем модель
model_path = 'models/full_churn_pipeline_retrained_cloud.pkl'
with open(model_path, 'rb') as f:
    pipeline = cloudpickle.load(f)

# Загружаем данные
df = pd.read_csv('data/new_data/new_data.csv')
df_labeled = df[df['Churn'].notna()].copy()

# ВАЖНО: НЕ удаляйте customerID! Пусть пайплайн сам с ним разбирается
# Просто возьмите ВСЕ колонки, кроме Churn и метаданных
exclude_cols = ['Churn', 'prediction', 'probability', 'timestamp', 'label_timestamp']
feature_columns = [col for col in df_labeled.columns if col not in exclude_cols]

X_raw = df_labeled[feature_columns].copy()

print(f"Raw data shape: {X_raw.shape}")
print(f"Raw columns: {list(X_raw.columns)}")

# Теперь предсказываем
try:
    y_pred = pipeline.predict(X_raw)
    print("Prediction successful!")
    print(f"Predictions: {y_pred}")
except Exception as e:
    print(f"Error: {e}")
    
    # Если ошибка, попробуем применить трансформации вручную
    print("\nTrying manual transformation...")
    X_transformed = X_raw.copy()
    for name, step in list(pipeline.named_steps.items())[:-1]:
        print(f"Applying {name}...")
        X_transformed = step.transform(X_transformed)
        print(f"  Shape: {X_transformed.shape}")
        if hasattr(X_transformed, 'columns'):
            print(f"  Columns: {list(X_transformed.columns)[:5]}...")
    
    print(f"\nFinal transformed shape: {X_transformed.shape}")
    
    # Предсказываем последним шагом
    model_step = pipeline.named_steps['model']
    y_pred = model_step.predict(X_transformed)
    print(f"Predictions: {y_pred}")

Raw data shape: (10, 20)
Raw columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Error: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 2 should be feature with name PhoneService (found Partner).

Trying manual transformation...
Applying totalcharges_cleaner...
  Shape: (10, 20)
  Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']...
Applying feature_engineer...
  Shape: (10, 26)
  Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']...
Applying categorical_encoder...
  Shape: (10, 18)
  Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure']...
Applying drop_redundant...
  Shape: (10, 15)
  Columns: ['gender', 'SeniorCitizen', 'Partner'

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 2 should be feature with name PhoneService (found Partner).

In [41]:
import pandas as pd
import cloudpickle
from pathlib import Path

def predict_safe(model_path, data_path):
    """
    Безопасное предсказание с автоматическим созданием всех колонок
    """
    # Загружаем модель
    with open(model_path, 'rb') as f:
        pipeline = cloudpickle.load(f)
    
    # Загружаем данные
    df = pd.read_csv(data_path)
    df_labeled = df[df['Churn'].notna()].copy()
    
    # Берем ВСЕ колонки, включая customerID
    # Убираем только Churn (целевая переменная)
    X_raw = df_labeled.drop('Churn', axis=1)
    
    print(f"Raw data shape: {X_raw.shape}")
    print(f"Raw columns: {list(X_raw.columns)}")
    
    # Применяем каждый трансформер по очереди и смотрим результат
    X = X_raw.copy()
    
    # Список трансформеров (все, кроме модели)
    transformers = ['totalcharges_cleaner', 'feature_engineer', 'categorical_encoder', 'drop_redundant', 'numerical_scaler']
    
    for name in transformers:
        if name in pipeline.named_steps:
            print(f"\nApplying {name}...")
            X = pipeline.named_steps[name].transform(X)
            print(f"  Shape: {X.shape}")
            if hasattr(X, 'columns'):
                print(f"  Columns: {list(X.columns)}")
            else:
                print(f"  Type: {type(X)}")
    
    # Получаем модель
    model = pipeline.named_steps['model']
    
    # Проверяем, что у нас есть все нужные колонки
    expected_cols = model.feature_names_
    print(f"\nModel expects {len(expected_cols)} features")
    
    if hasattr(X, 'columns'):
        # Проверяем, какие колонки отсутствуют
        missing_cols = set(expected_cols) - set(X.columns)
        extra_cols = set(X.columns) - set(expected_cols)
        
        if missing_cols:
            print(f"Missing columns: {missing_cols}")
            # Добавляем недостающие колонки со значением 0
            for col in missing_cols:
                X[col] = 0
        
        if extra_cols:
            print(f"Extra columns: {extra_cols}")
            # Удаляем лишние колонки
            X = X.drop(columns=extra_cols)
        
        # Переупорядочиваем колонки
        X = X[expected_cols]
        print(f"Final shape: {X.shape}")
        
        # Предсказываем
        y_pred = model.predict(X)
        print(f"\nPredictions: {y_pred}")
        return y_pred
    else:
        print("Cannot proceed - X is not DataFrame")
        return None

# Использование
predictions = predict_safe(
    'models/full_churn_pipeline_retrained_cloud.pkl',
    'data/new_data/new_data.csv'
)

Raw data shape: (10, 24)
Raw columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'prediction', 'probability', 'timestamp', 'label_timestamp']

Applying totalcharges_cleaner...
  Shape: (10, 24)
  Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'prediction', 'probability', 'timestamp', 'label_timestamp']

Applying feature_engineer...
  Shape: (10, 30)
  Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneServi